# Stage1 cap release local value information
Fixed 2018-2020 train / 2021-2022 evaluation, no held account.


In [1]:
import hashlib,json,os,sys,shlex,tempfile
from pathlib import Path
root=Path.cwd() if (Path.cwd()/'src').exists() else Path.cwd().parents[1];sys.path.insert(0,str(root))
import numpy as np,pandas as pd,qlib
from qlib.data import D
from qlib.data.dataset import DatasetH
from qlib.data.dataset.handler import DataHandlerLP
from qlib.data.dataset.loader import StaticDataLoader
from qlib.workflow import R
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from scipy.stats import spearmanr
from src.objective.relative_wealth import stationary_block_bootstrap_indices
from src.data.nas_market_data import archive_nas_recorder,materialize_nas_evidence,_ssh
card=root/'research/stage1_cap_release_value_information_20261023';plan=json.loads((card/'plan.json').read_text());src=plan['sources'];sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan['frozen'] and not (card/'run_state.json').exists()
for p,k in [(root/src['label_card']/'summary.json','label_summary_sha256'),(root/src['feature_card']/'summary.json','feature_summary_sha256'),(root/'src/objective/relative_wealth.py','bootstrap_module_sha256')]:assert sha(p)==src[k],str(p)
label_summary=json.loads((root/src['label_card']/'summary.json').read_text());feature_summary=json.loads((root/src['feature_card']/'summary.json').read_text())
assert label_summary['recorder_id']==src['label_recorder_id'] and label_summary['experiment_id']==src['label_experiment_id'] and label_summary['nas_readback']['verified']
assert feature_summary['recorder_id']==src['feature_recorder_id'] and feature_summary['nas_readback']['verified']
provider=(root/'data/qlib/full_path_tiered_allocation_20261013').resolve();ft=(root/src['feature_card']/'qlib_recorder').resolve();os.environ['MLFLOW_ALLOW_FILE_STORE']='true'
qlib.init(provider_uri=str(provider),region='us',exp_manager={'class':'MLflowExpManager','module_path':'qlib.workflow.expm','kwargs':{'uri':str(ft),'default_exp_name':Path(src['feature_card']).name}})
fr=R.get_recorder(experiment_name=Path(src['feature_card']).name,recorder_id=src['feature_recorder_id']);frame=fr.load_object('inputs/source_frame').reset_index(drop=True);path=fr.load_object('paths/reference_0')['replay'].reset_index(drop=True)
with R.uri_context(str((root/src['label_card']/'qlib_recorder').resolve())):
 lr=R.get_recorder(experiment_name=Path(src['label_card']).name,recorder_id=src['label_recorder_id']);labels=lr.load_object('analysis/labels').reset_index(drop=True)
assert len(frame)==len(path)==len(labels)==1061 and frame.trade_date.equals(path.trade_date) and frame.trade_date.equals(labels.trade_date)
cal=list(pd.to_datetime(D.calendar(start_time='2018-10-15',end_time='2022-12-30',freq='day')).date);assert cal==list(pd.to_datetime(frame.trade_date).dt.date)
original=plan['features']['original39'];extras=plan['features']['extra'];assert len(original)==39 and list(frame.columns[3:42])==original and extras==['raw_beta_minus_cap','risk_state']
assert (pd.to_datetime(frame.feature_available_at,utc=True)<=pd.to_datetime(labels.decision_at,utc=True)).all()
flat=frame[original].copy();flat['raw_beta_minus_cap']=labels.raw_beta_minus_cap.to_numpy(float);flat['risk_state']=path.risk_state.to_numpy(float)
flat['local_fee_net_advantage']=labels.local_fee_net_advantage.to_numpy(float);flat['decision_at']=labels.decision_at.to_numpy();flat['label_available_at']=labels.label_available_at.to_numpy();flat['eligible']=labels.eligible.to_numpy(bool)
flat.index=pd.MultiIndex.from_arrays([pd.to_datetime(frame.trade_date),np.repeat('SOXL_SOXX_CASH',len(frame))],names=['datetime','instrument'])
static=pd.concat({'feature':flat[original+extras],'label':flat[['local_fee_net_advantage']],'metadata':flat[['decision_at','label_available_at','eligible']]},axis=1)
dataset=DatasetH(handler=DataHandlerLP(instruments=None,start_time=flat.index.get_level_values('datetime').min(),end_time=flat.index.get_level_values('datetime').max(),data_loader=StaticDataLoader(config=static),infer_processors=[],learn_processors=[]),segments={'all':('2018-10-15','2022-12-30')})
prepared=dataset.prepare('all',col_set=['feature','label','metadata']);assert len(prepared)==1061 and list(prepared['feature'].columns)==original+extras
x=prepared['feature'];y=prepared['label']['local_fee_net_advantage'];meta=prepared['metadata'];dates=prepared.index.get_level_values('datetime');years=dates.year
eligible=meta['eligible'].astype(bool) & y.notna();cutoff=pd.Timestamp('2021-01-01T00:00:00',tz='America/New_York')
available=pd.to_datetime(meta['label_available_at'],utc=True);train=eligible & (years<=2020) & available.lt(cutoff.tz_convert('UTC'));evaluate=eligible & (years>=2021) & (years<=2022) & available.lt(pd.Timestamp('2023-01-01T00:00:00',tz='America/New_York').tz_convert('UTC'))
assert train.sum()>=plan['cohorts']['train_minimum_rows'] and all(int((evaluate & (years==z)).sum())>=plan['cohorts']['evaluation_minimum_each_year'] for z in (2021,2022))
assert not train.equals(evaluate) and not (train & evaluate).any()
state=card/'run_state.json';state.write_text(json.dumps({'status':'fit_started','plan_sha256':sha(card/'plan.json'),'fit_attempts_started':1},indent=2)+'\n')
model=Pipeline([('imputer',SimpleImputer(strategy='median')),('scale',StandardScaler()),('ridge',Ridge(alpha=10.0))]);model.fit(x.loc[train],y.loc[train]);pred=model.predict(x.loc[evaluate]);control=float(y.loc[train].mean())
eval_dates=dates[evaluate];eval_years=eval_dates.year;truth=y.loc[evaluate].to_numpy(float);gain=np.square(truth-control)-np.square(truth-pred)
per_year=[]
for yr in (2021,2022):
 mask=np.asarray(eval_years==yr);rho=float(spearmanr(pred[mask],truth[mask]).statistic);per_year.append({'year':yr,'rows':int(mask.sum()),'paired_MSE_improvement':float(gain[mask].mean()),'Spearman':rho})
full_calendar=(years>=2021)&(years<=2022);full_diffs=np.zeros(int(full_calendar.sum()));full_mask=np.asarray(evaluate[full_calendar]);full_diffs[full_mask]=gain
b=plan['information_gate']['bootstrap'];idx=stationary_block_bootstrap_indices(observation_count=len(full_diffs),path_length=len(full_diffs),path_count=b['draws'],expected_block_length=b['mean_block_sessions'],seed=b['seed']);counts=full_mask[idx].sum(axis=1);assert (counts>0).all();draws=full_diffs[idx].sum(axis=1)/counts
lower=float(np.quantile(draws,b['one_sided_alpha']));upper=float(np.quantile(draws,1-b['one_sided_alpha']))
gates={'full_paired_MSE_improvement_positive':bool(gain.mean()>0),'each_2021_2022_paired_MSE_improvement_positive':all(z['paired_MSE_improvement']>0 for z in per_year),'each_2021_2022_Spearman_positive':all(z['Spearman']>0 for z in per_year),'full_calendar_stationary_block_corrected_lower_positive':bool(lower>0)}
passed=all(gates.values());result={'decision':'conditional_information_pass_needs_full_0bp_account' if passed else 'not_promoted_fixed_cap_release_value_information_failed','objective_achieved':False,'train_rows':int(train.sum()),'evaluation_rows':int(evaluate.sum()),'evaluation_calendar_rows':len(full_diffs),'train_label_mean_control':control,'candidate_MSE':float(np.square(truth-pred).mean()),'control_MSE':float(np.square(truth-control).mean()),'full_paired_MSE_improvement':float(gain.mean()),'per_year':per_year,'bootstrap':{'draws':b['draws'],'mean_block_sessions':b['mean_block_sessions'],'seed':b['seed'],'one_sided_alpha':b['one_sided_alpha'],'lower':lower,'upper':upper,'calendar_rows':len(full_diffs),'scored_rows':int(full_mask.sum()),'indices_sha256':hashlib.sha256(idx.tobytes()).hexdigest()},'gates':gates,'new_model_fits':1,'new_bootstraps':1,'new_native_accounts':0,'independent_forward_days':0,'historical_independence':False}
(card/'result.json').write_text(json.dumps(result,ensure_ascii=False,indent=2)+'\n')
evaluation=pd.DataFrame({'trade_date':eval_dates,'year':eval_years,'truth':truth,'candidate_prediction':pred,'control_prediction':control,'paired_MSE_improvement':gain})
with R.start(experiment_name=plan['experiment'],uri=str(card/'qlib_recorder')):
 rec=R.get_recorder();rec.save_objects(artifact_path='inputs',plan=plan,dataset=dataset);rec.save_objects(artifact_path='models',candidate=model);rec.save_objects(artifact_path='analysis',evaluation=evaluation,result=result,bootstrap_draws=draws);rec.log_metrics(train_rows=int(train.sum()),evaluation_rows=int(evaluate.sum()),paired_MSE_improvement=float(gain.mean()),bootstrap_lower=lower,new_model_fits=1,new_native_accounts=0);rid=rec.id;eid=str(rec.experiment_id)
summary={'experiment':plan['experiment'],'decision':result['decision'],'objective_achieved':False,'plan_sha256':sha(card/'plan.json'),'result':'result.json','recorder_id':rid,'experiment_id':eid,'source_label_recorder_id':src['label_recorder_id'],'new_model_fits':1,'new_bootstraps':1,'new_native_accounts':0,'artifact_policy':{'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['inputs/dataset','models/candidate','analysis/evaluation','analysis/bootstrap_draws']}}
binding=json.loads((root/src['label_card']/'nas_binding.json').read_text());binding.pop('evidence',None);binding.update(source_id=plan['experiment'],source_plan='plan.json',as_of='2022-12-30');(card/'nas_binding.json').write_text(json.dumps(binding,ensure_ascii=False,indent=2)+'\n')
evidence=archive_nas_recorder(card/'nas_binding.json',workspace_root=root,card=card,recorder_id=rid,experiment_id=eid);remote=_ssh(binding['ssh'],'sha256sum '+shlex.quote(evidence['nas_path'])).split()[0];assert remote==evidence['sha256']
with tempfile.TemporaryDirectory(prefix='soxl-cap-info-readback-') as temp:
 restored=Path(temp)/'card';restored.mkdir();materialize_nas_evidence(card/'nas_binding.json',card=restored);rel=Path('qlib_recorder')/eid/rid/'artifacts/analysis/evaluation';assert sha(card/rel)==sha(restored/rel)
summary['nas_evidence']=evidence;summary['nas_readback']={'verified':True,'sha256':remote,'scoped_restored_evaluation_sha256':sha(card/rel)}
(card/'summary.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2)+'\n');state.write_text(json.dumps({'status':'completed','plan_sha256':sha(card/'plan.json'),'fit_attempts_completed':1,'bootstrap_attempts_completed':1,'recorder_id':rid},indent=2)+'\n')
print(json.dumps({'result':result,'recorder_id':rid,'NAS_sha256':remote},ensure_ascii=False))

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[30623:MainThread](2026-10-23 08:24:02,388) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[30623:MainThread](2026-10-23 08:24:02,390) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[30623:MainThread](2026-10-23 08:24:02,390) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/full_path_tiered_allocation_20261013')}


[30623:MainThread](2026-10-23 08:24:02,465) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | Loading data Done


...

[30623:MainThread](2026-10-23 08:24:02,564) WARNING - qlib.workflow - [expm.py:231] - No valid experiment found. Create a new experiment with name stage1_cap_release_value_information_20261023.


[30623:MainThread](2026-10-23 08:24:02,565) INFO - qlib.workflow - [exp.py:258] - Experiment 658507532621177279 starts running ...


[30623:MainThread](2026-10-23 08:24:02,710) INFO - qlib.workflow - [recorder.py:345] - Recorder ed8483b445e74e45b75281b5bd67cbd3 starts running under Experiment 658507532621177279 ...


[30623:MainThread](2026-10-23 08:24:03,822) INFO - qlib.timer - [log.py:127] - Time cost: 0.002s | waiting `async_log` Done


{"result": {"decision": "not_promoted_fixed_cap_release_value_information_failed", "objective_achieved": false, "train_rows": 239, "evaluation_rows": 301, "evaluation_calendar_rows": 503, "train_label_mean_control": 0.00019368167887498505, "candidate_MSE": 8.199876818417991e-06, "control_MSE": 6.330007046203221e-06, "full_paired_MSE_improvement": -1.8698697722147687e-06, "per_year": [{"year": 2021, "rows": 56, "paired_MSE_improvement": -2.1892962579359416e-06, "Spearman": -0.040738209159261796}, {"year": 2022, "rows": 245, "paired_MSE_improvement": -1.7968580040499292e-06, "Spearman": 0.061264473802315825}], "bootstrap": {"draws": 10000, "mean_block_sessions": 63, "seed": 20261023, "one_sided_alpha": 0.00078125, "lower": -3.3744481266679513e-06, "upper": -7.15163652499325e-07, "calendar_rows": 503, "scored_rows": 301, "indices_sha256": "2bf4bb3159e4081eac4d417885bad5a1d199df31a3379ae20031b41bf3ddb9b4"}, "gates": {"full_paired_MSE_improvement_positive": false, "each_2021_2022_paired_MSE

# Stage1 风险上限释放局部价值：固定跨年信息门失败

按事前冻结的 41 项输入（原 Stage1 39 项，加账户当时的 beta 上限缺口和风险状态），只在 2021 年初以前成熟的 **239** 个标签上拟合一次 Ridge；**2021/2022 共 301** 个标签按原口径评价，另保留 503 个完整交易日的区块结构。对照为训练标签均值，不使用 2023+ 结果，也没有策略账户。

候选 MSE **0.0000081999**，高于对照 **0.0000063300**；配对改善为 **-0.0000018699**，2021 与 2022 分别为负。2021 排序 Spearman **-0.0407**；冻结区块检验的校正下界 **-0.0000033744**，上界也为负。四项信息门全部失败。

只读预测回查发现年度校准方向也相反：2021 实际局部均值 +0.000541，模型均值 −0.000466；2022 实际 −0.000123，模型均值 +0.000541。这是预测失败的具体位置，不证明某个市场状态有因果作用，也不能据此反转模型、只选年份或改阈值。固定版本 **`not_promoted`**，不运行完整 Stage1 候选账户。

原生 Qlib Recorder `ed8483b445e74e45b75281b5bd67cbd3` 保存模型、预测和 10000 次完整日历区块抽样；NAS SHA-256 `3b48813598c3faf012db134090ba3a61d9196da36c2574d65d6033920f1ae974` 已远端核对并限定恢复评价对象。共 1 次新拟合、1 次预定 bootstrap、0 新行情请求、0 候选账户。SOXX 硬门、2/5bp 和独立前向证据均未达成；Goal 继续寻找实质不同的决策时机制。
